# Scalar automatic differentiation

A learning implementation following Andrej Karpathy's micrograd tutorial. The graph records local derivatives; a reverse topological traversal accumulates gradients using the chain rule. The Graphviz helper below is copied from the tutorial.

The training example uses a fixed seed, at most 2,000 steps, and squared-error loss. It is a small demonstration, not a benchmark.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)


# VALUE CLASS
class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __radd__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __rmul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = math.tanh(x)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), "exp")

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f"**{other}")

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


# GRAPHVIZ (JUST COPY-PASTE)
from graphviz import Digraph


def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root):
    dot = Digraph(format="svg", graph_attr={"rankdir": "LR"})  # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(
            name=uid,
            label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad),
            shape="record",
        )
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name=uid + n._op, label=n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot


# NEURAL NETS


class Neuron:
    def __init__(self, nin):
        self.w = [Value(np.random.randn() / math.sqrt(nin)) for i in range(nin)]
        self.b = Value(0.0)

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        params = []
        for n in self.neurons:
            ps = n.parameters()
            params.extend(ps)
        return params


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        params = []
        for n in self.layers:
            ps = n.parameters()
            params.extend(ps)
        return params


# LOSS FUNCTION
x = [[1.0, 2.0], [2.0, 3.0], [3.0, 4.0], [4.0, 5.0]]
n = MLP(2, [4, 4, 1])
y = [n(xi) for xi in x]
ypred = [1.0, 2.0, 3.0, 4.0]
loss = sum([(yp - yi) ** 2 for yi, yp in zip(y, ypred)])
n.parameters()


[Value(data=0.35122994590557494), Value(data=-0.09776762495416377), Value(data=0.0), Value(data=0.4579849573878012), Value(data=1.0769447394156884), Value(data=0.0), Value(data=-0.16557143910458558), Value(data=-0.1655598299851483), Value(data=0.0), Value(data=1.1166720907819767), Value(data=0.5426583011020832), Value(data=0.0), Value(data=-0.23473719296747605), Value(data=0.27128002179298233), Value(data=-0.23170884640623113), Value(data=-0.23286487678512843), Value(data=0.0), Value(data=0.12098113578301706), Value(data=-0.956640122328899), Value(data=-0.8624589162565164), Value(data=-0.28114376462048635), Value(data=0.0), Value(data=-0.5064155601672119), Value(data=0.15712366629763694), Value(data=-0.45401203776060545), Value(data=-0.7061518506676457), Value(data=0.0), Value(data=0.732824384460777), Value(data=-0.11288815024326783), Value(data=0.03376410234396192), Value(data=-0.7123740931067284), Value(data=0.0), Value(data=-0.27219136226259133), Value(data=0.05546129485493304), Val

In [2]:
x = [[1, 2], [2, 3], [3, 4], [5, 6]]
n = MLP(2, [4, 4, 1])
ypred = [n(xi) for xi in x]
y = [1.0, -1.0, -1.0, 1.0]
loss = sum([(yi - yp) ** 2 for yi, yp in zip(y, ypred)])
loss


Value(data=4.214626445617609)

In [3]:
ypred


[Value(data=-0.2194850722970934), Value(data=-0.18364860015280293), Value(data=-0.17414836235595776), Value(data=-0.1743177034475537)]

In [4]:
for step in range(2000):
    # forward pass
    ypred = [n(xi) for xi in x]
    loss = sum([(yi - yp) ** 2 for yi, yp in zip(y, ypred)])

    # 0grad
    for p in n.parameters():
        p.grad = 0

    # backward pass
    loss.backward()

    # gradient descent
    for p in n.parameters():
        p.data += -0.01 * p.grad
    if step % 200 == 0:
        print(f"step {step}: loss={loss.data:.6f}")
    if loss.data < 0.0001:
        break
print(f"Final loss: {loss.data:.6f}")


step 0: loss=4.214626
step 200: loss=3.871911
step 400: loss=3.084437
step 600: loss=2.493335
step 800: loss=1.028544
step 1000: loss=0.012967
step 1200: loss=0.008020
step 1400: loss=0.005871
step 1600: loss=0.004633
step 1800: loss=0.003822
Final loss: 0.003251


In [5]:
ypred


[Value(data=0.9749711776642636), Value(data=-0.9765096026370015), Value(data=-0.9671312216114296), Value(data=0.9684979203916152)]